# Arya E4B Action Adapter — Colab QLoRA

This notebook starts a **Gemma E4B-specific Arya Action IR adapter**.

It does **not** use E2B. It does not replace Arya's deterministic compiler or executor.

```text
Deterministic compiler -> validated Action IR -> E4B Action Adapter on ambiguity/complex planning
```

> Important: the LiteRT-LM `.litertlm` file used by the Android APK is an inference artifact. Fine-tuning must use a trainable Transformers checkpoint for the equivalent E4B base model, then evaluate and package the adapter separately.


## Before running

1. Colab: **Runtime → Change runtime type → GPU**.
2. Hugging Face: accept the base model terms if required.
3. Colab left sidebar: **Secrets → add `HF_TOKEN`**.
4. Set `MODEL_ID` below to the exact trainable E4B Transformers checkpoint you are licensed to use.
5. Start with `SMOKE_ONLY=True`. The seed corpus is a pipeline test, not enough data for a production adapter.

Never paste Hugging Face tokens into notebook cells, GitHub, chat or training data.


In [ ]:
# Runtime and model configuration
from pathlib import Path

# REQUIRED: replace with the trainable E4B/4B Transformers checkpoint.
# Do not use the Android .litertlm inference file here.
MODEL_ID = "google/gemma-4-E4B-it"

GITHUB_DATASET_URL = (
    "https://raw.githubusercontent.com/asmoia/arya-agent/main/"
    "models/arya-e4b-action/dataset/seed.fa-en.jsonl"
)

# Conservative defaults for free T4/P100 experiments.
SEQ_LEN = 256
LORA_R = 4
LORA_ALPHA = 8
LEARNING_RATE = 1e-4
MAX_STEPS = 1
SMOKE_ONLY = True
# One accumulation step makes the first smoke run fast and easier to diagnose.
# Increase only after the autograd preflight passes.
GRADIENT_ACCUMULATION_STEPS = 1 if SMOKE_ONLY else 8
ALLOW_TINY_SEED_TRAINING = False

WORKDIR = Path("/content/arya_e4b_action")
DATA_PATH = WORKDIR / "seed.fa-en.jsonl"
OUTPUT_DIR = WORKDIR / "adapter"
WORKDIR.mkdir(parents=True, exist_ok=True)
print({"model_id": MODEL_ID, "smoke_only": SMOKE_ONLY, "workdir": str(WORKDIR)})


In [ ]:
# GPU preflight. Do not continue with a CPU runtime.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
assert torch.cuda.is_available(), "Enable GPU in Colab: Runtime -> Change runtime type -> GPU"
props = torch.cuda.get_device_properties(0)
vramb = props.total_memory / (1024 ** 3)
print({"gpu": props.name, "vram_gb": round(vramb, 2), "cuda": torch.version.cuda})
if vramb < 15:
    print("WARNING: under 15GB VRAM. Keep 4-bit QLoRA, batch=1 and short sequence; OOM is possible.")


In [ ]:
# Install the training stack. Re-run after a Colab runtime restart.
!pip -q install -U "transformers>=4.51" "accelerate>=1.4" "peft>=0.14" "trl>=0.15" "datasets>=3.3" bitsandbytes sentencepiece


In [ ]:
# Read the HF token from Colab Secrets and download the validated seed corpus.
from google.colab import userdata
from huggingface_hub import login
import urllib.request

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Create a Colab Secret named HF_TOKEN; do not paste it in this notebook."
login(token=HF_TOKEN, add_to_git_credential=False)

urllib.request.urlretrieve(GITHUB_DATASET_URL, DATA_PATH)
print({"dataset": str(DATA_PATH), "bytes": DATA_PATH.stat().st_size})


In [ ]:
# Validate the Action IR corpus before GPU work.
# This is the same logical gate that runs in GitHub Actions.
import json

ALLOWED_ACTIONS = {
    "open_app", "open_messaging_chat", "prepare_message", "commit_message",
    "search_browser", "read_visible_content", "summarize_visible_content",
    "get_device_info", "take_screenshot", "system_back", "system_home",
    "wait_for_ui", "clarify_contact", "finish",
}
rows = [json.loads(line) for line in DATA_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
assert len({row["id"] for row in rows}) == len(rows), "Dataset IDs must be unique"
for row in rows:
    target = row["target"]
    assert target["version"] == "arya.action.v1"
    assert target["route"] in {"execute", "clarify", "refuse", "escalate"}
    assert isinstance(target["plan"], list) and len(target["plan"]) <= 8
    for step in target["plan"]:
        assert step["action"] in ALLOWED_ACTIONS
    if target["route"] in {"clarify", "refuse"}:
        assert not target["plan"]
    if target["risk_level"] == "blocked":
        assert target["route"] == "refuse"
print({"validated_rows": len(rows), "smoke_only": SMOKE_ONLY})


In [ ]:
# Convert Action IR examples to a compact supervised format.
from datasets import Dataset

SYSTEM = """You are Arya E4B Action Adapter. Return exactly one valid ary.action.v1 JSON object. Do not reveal reasoning. Do not invent actions. Respect risk and confirmation semantics."""

def render_example(row):
    user = json.dumps(row["input"], ensure_ascii=False, separators=(",", ":"))
    target = json.dumps(row["target"], ensure_ascii=False, separators=(",", ":"))
    return {"text": f"<system>\n{SYSTEM}\n</system>\n<user>\n{user}\n</user>\n<assistant>\n{target}\n</assistant>"}

train_rows = [render_example(row) for row in rows]
if len(train_rows) < 500 and not (SMOKE_ONLY or ALLOW_TINY_SEED_TRAINING):
    raise RuntimeError("Seed data is too small for real training. Expand validated corpus to >=500 rows or explicitly set ALLOW_TINY_SEED_TRAINING.")
train_dataset = Dataset.from_list(train_rows)
print(train_dataset[0]["text"][:500])


## If the previous run OOMed

1. Use **Runtime → Restart session** before retrying; an OOM can leave cached CUDA allocations.
2. Confirm the current notebook contains the manual-freeze cell below and does **not** call `prepare_model_for_kbit_training`.
3. Keep the free-GPU defaults:

```python
SEQ_LEN = 256
LORA_R = 4
LORA_ALPHA = 8
```

4. If model loading itself still OOMs, do not force it. A free 14–16GB GPU is insufficient for that checkpoint/session combination; use a larger GPU or a smaller sequence/data smoke path. Do not switch Arya's architecture to E2B as an OOM workaround.


## E4B target discovery

Gemma E4B checkpoints can expose wrapped projection modules in vision and language branches. Before PEFT attaches LoRA, the notebook runs one frozen text probe with forward hooks and selects only inner `Linear4bit` projections that actually executed. This prevents attaching adapters to inactive wrapper paths that produce no gradient.


In [ ]:
# Load E4B in 4-bit QLoRA mode.
# If this OOMs on free Colab, reduce SEQ_LEN first. Do not switch the project to E2B.
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

compute_dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=quant,
    device_map={"": 0},
    torch_dtype=compute_dtype,
)
model.config.use_cache = False

# Do NOT call prepare_model_for_kbit_training() here. On E4B in free Colab
# sessions, recent PEFT versions may cast non-4bit tensors to fp32 and allocate
# ~10GB at once. Freeze base weights manually instead; LoRA will re-enable only
# adapter parameters below.
for parameter in model.parameters():
    parameter.requires_grad_(False)

# Discover only trainable 4-bit projections used by a real TEXT forward.
# Gemma4 text projections can be direct Linear4bit modules (`q_proj`), while
# multimodal clipped wrappers expose inner modules (`q_proj.linear`).
projection_roots = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
candidates = {}
for module_name, module in model.named_modules():
    if module.__class__.__name__ != "Linear4bit":
        continue
    leaf = module_name.rsplit(".", 1)[-1]
    parent = module_name.rsplit(".", 2)[-2] if module_name.count(".") >= 2 and leaf == "linear" else None
    if leaf in projection_roots or parent in projection_roots:
        candidates[module_name] = module

assert candidates, "No candidate Linear4bit projection modules found."
executed_base_targets = set()
hooks = [
    module.register_forward_hook(
        lambda _module, _inputs, _output, name=name: executed_base_targets.add(name)
    )
    for name, module in candidates.items()
]
probe = tokenizer(
    "Arya Action IR probe: open Telegram",
    return_tensors="pt",
    truncation=True,
    max_length=min(SEQ_LEN, 64),
)
probe = {name: tensor.to(model.device) for name, tensor in probe.items()}
model.eval()
with torch.no_grad():
    _ = model(**probe)
for hook in hooks:
    hook.remove()

target_modules = sorted(executed_base_targets)
if not target_modules:
    print({"candidate_count": len(candidates), "candidate_names": sorted(candidates)[:40]})
    raise RuntimeError("No candidate Linear4bit projection executed during text probe. Stop before LoRA and send this diagnostic.")
print({
    "candidate_target_count": len(candidates),
    "executed_lora_target_count": len(target_modules),
    "executed_lora_targets": target_modules[:20],
})

lora = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)
model = get_peft_model(model, lora)

# Explicitly verify that PEFT re-enabled adapter weights after base freeze.
lora_parameters = [(name, parameter) for name, parameter in model.named_parameters() if "lora_" in name]
trainable_lora = [(name, parameter) for name, parameter in lora_parameters if parameter.requires_grad]
assert trainable_lora, "No trainable LoRA parameters found. Stop before Trainer and inspect target_modules."
print({"trainable_lora_tensors": len(trainable_lora), "first_trainable": [name for name, _ in trainable_lora[:12]]})

# K-bit E4B needs input embeddings connected to the adapter graph, even during
# smoke mode. Do this after PEFT wraps the model.
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
else:
    def _require_input_grad(_, __, output):
        output.requires_grad_(True)
    model.get_input_embeddings().register_forward_hook(_require_input_grad)

# Disable checkpointing only for the first smoke diagnostic. Re-enable after
# the preflight passes when increasing steps/sequence length.
if not SMOKE_ONLY:
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except TypeError:
        model.gradient_checkpointing_enable()
else:
    try:
        model.gradient_checkpointing_disable()
    except Exception:
        pass

model.print_trainable_parameters()


## Autograd preflight

Before `Trainer` starts, the next cell performs a real forward/backward pass on one row and verifies that at least one `lora_*` parameter receives a gradient. If this fails, stop there: the error identifies the model/PEFT target issue immediately instead of wasting a training session.


In [ ]:
# Train with the stable Transformers Trainer API.
# We intentionally avoid TRL SFTTrainer because its constructor changes
# between Colab-installed TRL releases.
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=SEQ_LEN, padding=False)

tokenized_dataset = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing Arya Action IR corpus",
)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Real autograd preflight. It must succeed before Trainer starts.
executed_lora_modules = set()
hooks = []
for module_name, module in model.named_modules():
    if hasattr(module, "lora_A"):
        hooks.append(module.register_forward_hook(
            lambda _module, _inputs, _output, name=module_name: executed_lora_modules.add(name)
        ))
smoke_batch = collator([tokenized_dataset[0]])
smoke_batch = {name: tensor.to(model.device) for name, tensor in smoke_batch.items()}
model.train()
smoke_output = model(**smoke_batch)
for hook in hooks:
    hook.remove()

if smoke_output.loss is None or not smoke_output.loss.requires_grad:
    print({
        "loss": None if smoke_output.loss is None else float(smoke_output.loss.detach().cpu()),
        "loss_requires_grad": None if smoke_output.loss is None else smoke_output.loss.requires_grad,
        "configured_lora_targets": target_modules[:20],
        "executed_lora_module_count": len(executed_lora_modules),
        "executed_lora_modules": sorted(executed_lora_modules)[:20],
        "trainable_lora": [name for name, _ in trainable_lora[:20]],
    })
    raise RuntimeError("Autograd preflight failed before Trainer. Send the dictionary above; do not run trainer.train().")

smoke_output.loss.backward()
grad_names = [name for name, parameter in trainable_lora if parameter.grad is not None]
model.zero_grad(set_to_none=True)
torch.cuda.empty_cache()
if not grad_names:
    print({
        "loss_requires_grad": smoke_output.loss.requires_grad,
        "configured_lora_targets": target_modules[:20],
        "executed_lora_module_count": len(executed_lora_modules),
        "executed_lora_modules": sorted(executed_lora_modules)[:20],
        "trainable_lora": [name for name, _ in trainable_lora[:20]],
        "lora_grad_names": grad_names,
    })
    raise RuntimeError("LoRA modules were not on the text loss path. Stop before Trainer and send this dictionary.")
print({"autograd_preflight": "passed", "executed_lora_module_count": len(executed_lora_modules), "lora_grad_tensors": len(grad_names)})

steps = MAX_STEPS if SMOKE_ONLY else max(MAX_STEPS, 200)
args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    max_steps=steps,
    logging_steps=1,
    save_steps=max(1, steps),
    save_total_limit=2,
    # Avoid CUDA GradScaler hiding a detached adapter graph in smoke mode.
    fp16=False,
    bf16=False,
    report_to="none",
    optim="paged_adamw_8bit",
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset,
    data_collator=collator,
    processing_class=tokenizer,
)
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print({"adapter_dir": str(OUTPUT_DIR), "steps": steps, "tokenized_rows": len(tokenized_dataset)})


In [ ]:
# Save the adapter and a reproducibility manifest to Google Drive.
from google.colab import drive
import shutil

drive.mount("/content/drive")
DRIVE_OUTPUT = Path("/content/drive/MyDrive/arya_e4b_action_adapter")
DRIVE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
if DRIVE_OUTPUT.exists():
    shutil.rmtree(DRIVE_OUTPUT)
shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT)

manifest = {
    "base_model": MODEL_ID,
    "adapter": "arya-e4b-action",
    "action_ir": "arya.action.v1",
    "sequence_length": SEQ_LEN,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "smoke_only": SMOKE_ONLY,
    "seed_rows": len(rows),
}
(DRIVE_OUTPUT / "arya_training_manifest.json").write_text(json.dumps(manifest, indent=2))
print({"saved_to": str(DRIVE_OUTPUT), "manifest": manifest})


## Next step after a successful run

Do **not** package this adapter into the Android APK yet.

First:

1. expand validated corpus beyond the seed;
2. evaluate held-out Action IR validity, route/slot correctness and safety;
3. compare base E4B vs adapter on real Android devices;
4. record RAM mode, cold/warm latency, thermal state and native-generation count;
5. only then decide how to merge/package for LiteRT.
